In [ ]:
# In[1]:

import pandas as pd

# Read files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Basic per-file summaries: sorted unique cmdb_id and row counts
def file_summary(df, name):
    unique_cmdb = sorted(df['cmdb_id'].dropna().unique())
    return {'file': name, 'row_count': df.shape[0], 'unique_cmdb_ids': unique_cmdb}

per_file_summary = pd.DataFrame([
    file_summary(metric_df, 'metric.csv'),
    file_summary(trace_df, 'trace.csv'),
    file_summary(log_df, 'log.csv'),
    file_summary(error_df, 'error_logs.csv'),
])

# --- metric.csv: unique kpi_name values and global P95 by (cmdb_id, kpi_name)
metric_kpi_names = pd.Series(sorted(metric_df['kpi_name'].dropna().unique()), name='kpi_name')
# ensure numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)
metric_p95_top20 = metric_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
metric_sample = metric_df[['timestamp','cmdb_id','kpi_name','value']].head(5)

# --- trace.csv: unique trace_name values and global P95 by (cmdb_id, trace_name)
trace_names = pd.Series(sorted(trace_df['trace_name'].dropna().unique()), name='trace_name')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)
trace_p95_top20 = trace_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
trace_sample = trace_df[['timestamp','cmdb_id','trace_name','value']].head(5)

# --- log.csv: unique log_name values and global P95 by (cmdb_id, log_name)
log_names = pd.Series(sorted(log_df['log_name'].dropna().unique()), name='log_name')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)
log_p95_top20 = log_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
log_sample = log_df[['timestamp','cmdb_id','log_name','value']].head(5)

# --- error_logs.csv: unique cmdb_id and sample rows
error_unique_cmdb = pd.Series(sorted(error_df['cmdb_id'].dropna().unique()), name='cmdb_id')
error_sample = error_df.head(5)

# Compact outputs (as requested). Returning these variables (IPython will display them).
per_file_summary, metric_kpi_names, metric_p95_top20, metric_sample, trace_names, trace_p95_top20, trace_sample, log_names, log_p95_top20, log_sample, error_unique_cmdb, error_sample

```
Out[1]:
```
Summary (plain English):

- Files loaded and timestamps parsed as UTC.

- Row counts and cmdb_id presence:
  - metric.csv: 9,073 rows; many cmdb_id values (services like ts-admin-*, ts-travel-*, ts-order-*, etc.).
  - trace.csv: 18,392 rows; many trace_name types (220 distinct trace_name values) and many cmdb_id values.
  - log.csv: 2,186 rows; several cmdb_id values.
  - error_logs.csv: 0 rows (empty); no cmdb_id entries.

- metric.csv findings:
  - Unique kpi_name values found: cpu, diskio, error, latency-50, latency-90, mem, socket, workload.
  - Global P95 (grouped by cmdb_id and kpi_name): the highest P95s are dominated by the mem KPI. Top entries include:
    - ts-voucher-mysql, mem — p95 ≈ 6.003794e+08
    - ts-food-mysql, mem — p95 ≈ 5.998060e+08
    - multiple application services (e.g., ts-travel2-service, ts-travel-service, ts-order-service, ts-route-service, etc.) have mem p95 values around 2.4–2.8e+08.
  - Up to 5 sample metric rows were returned (examples are data for ts-admin-basic-info-service at 2024-01-24 10:29 UTC for cpu, diskio, error, latency-50, latency-90).

- trace.csv findings:
  - 220 unique trace_name values.
  - Global P95 (grouped by cmdb_id and trace_name): top results are mainly trace row_count metrics (large counts), e.g.:
    - ts-station-service trace.from_* / trace.to_* .row_count — p95 ≈ 22956.0
    - ts-travel2-service .row_count — p95 ≈ 2728.8
    - ts-train-service .row_count — p95 ≈ 2581.6
    - additional services with high row_count p95s follow.
  - Up to 5 sample trace rows were returned (examples at 2024-01-24 10:29 UTC).

- log.csv findings:
  - Unique log_name values: log.error_count and log.row_count.
  - Global P95 (grouped by cmdb_id and log_name): top log.row_count p95s include:
    - ts-basic-service — p95 = 2575.00
    - ts-travel2-service — p95 = 2401.20
    - ts-seat-service — p95 = 1909.60
    - ts-travel-service — p95 = 1890.20
    - ts-route-service — p95 = 1037.80
  - Up to 5 sample log rows were returned (examples at 2024-01-24 10:29 UTC).

- error_logs.csv: empty (no sample rows or cmdb_id values).

Key takeaways:
- Memory (mem) metric shows the largest P95 values across many services (including DB instances ts-voucher-mysql and ts-food-mysql), indicating memory is the dominant high-value KPI in this dataset.
- Trace row_count and log.row_count show very large P95s for certain services (e.g., ts-station-service, ts-basic-service, ts-travel2-service), indicating high request/trace/log volumes for those services.
- No error log entries were available in error_logs.csv.

If you want, I can:
- produce a compact table of the top N offending services per KPI,
- or focus on a specific service to show its full time series and P95 calculation details.

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count                                    unique_cmdb_ids
0      metric.csv       9073  [ts-admin-basic-info-service, ts-admin-order-s...
1       trace.csv      18392  [root, ts-admin-basic-info-service, ts-admin-t...
2         log.csv       2186  [ts-admin-basic-info-service, ts-admin-travel-...
3  error_logs.csv          0                                                 [], 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: kpi_name, dtype: object,                          cmdb_id kpi_name           p95
0               ts-voucher-mysql      mem  6.003794e+08
1                  ts-food-mysql      mem  5.998060e+08
2             ts-travel2-service      mem  2.788872e+08
3              ts-travel-service      mem  2.736922e+08
4               ts-order-service      mem  2.710809e+08
5               ts-route-service      mem  2.708520e+08
6         ts-order-other-service      mem  2.707900e+08
7             ts-station-service      mem  2.691350e+08
8   ts-verification-code-service      mem  2.631393e+08
9                ts-food-service      mem  2.631299e+08
10           ts-food-map-service      mem  2.567888e+08
11     ts-inside-payment-service      mem  2.562440e+08
12              ts-price-service      mem  2.550278e+08
13              ts-train-service      mem  2.549171e+08
14           ts-security-service      mem  2.531701e+08
15             ts-config-service      mem  2.509413e+08
16            ts-consign-service      mem  2.495456e+08
17           ts-contacts-service      mem  2.484691e+08
18              ts-basic-service      mem  2.473230e+08
19      ts-consign-price-service      mem  2.470978e+08,                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service         cpu      0.417604
1 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service      diskio  52074.875307
2 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service       error      0.000000
3 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service  latency-50      0.017466
4 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service  latency-90      0.024196, 0                          trace.from_root.duration_mean
1                           trace.from_root.duration_p95
2                             trace.from_root.error_rate
3                              trace.from_root.row_count
4      trace.from_ts-admin-basic-info-service.duratio...
                             ...                        
215                trace.to_ts-travel2-service.row_count
216               trace.to_ts-user-service.duration_mean
217                trace.to_ts-user-service.duration_p95
218                  trace.to_ts-user-service.error_rate
219                   trace.to_ts-user-service.row_count
Name: trace_name, Length: 220, dtype: object,                   cmdb_id                                  trace_name      p95
0      ts-station-service     trace.from_ts-station-service.row_count  22956.0
1      ts-station-service       trace.to_ts-station-service.row_count  22956.0
2      ts-travel2-service     trace.from_ts-travel2-service.row_count   2728.8
3      ts-travel2-service       trace.to_ts-travel2-service.row_count   2728.8
4        ts-train-service         trace.to_ts-train-service.row_count   2581.6
5        ts-train-service       trace.from_ts-train-service.row_count   2581.6
6        ts-route-service       trace.from_ts-route-service.row_count   2075.6
7        ts-route-service         trace.to_ts-route-service.row_count   2075.6
8        ts-basic-service         trace.to_ts-basic-service.row_count   1864.6
9        ts-basic-service       trace.from_ts-basic-service.row_count   1864.6
10      ts-travel-service        trace.to_ts-travel-service.row_count   1676.4
11      ts-travel-service      trace.from_ts-travel-service.row_count   1676.4
12       ts-basic-service       trace.to_ts-station-service.row_count   1439.0
13     ts-station-service       trace.from_ts-basic-service.row_count   1439.0
14        ts-seat-service        trace.from_ts-seat-service.row_count   1087.6
15        ts-seat-service          trace.to_ts-seat-service.row_count   1087.6
16  ts-ticketinfo-service         trace.to_ts-basic-service.row_count   1012.6
17  ts-ticketinfo-service  trace.from_ts-ticketinfo-service.row_count   1012.6
18  ts-ticketinfo-service    trace.to_ts-ticketinfo-service.row_count   1012.6
19       ts-basic-service  trace.from_ts-ticketinfo-service.row_count   1012.6,                   timestamp cmdb_id                                         trace_name      value
0 2024-01-24 10:29:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_...   0.009974
1 2024-01-24 10:29:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_p95   0.011652
2 2024-01-24 10:29:00+00:00    root    trace.to_ts-admin-basic-info-service.error_rate   0.000000
3 2024-01-24 10:29:00+00:00    root     trace.to_ts-admin-basic-info-service.row_count  15.000000
4 2024-01-24 10:29:00+00:00    root     trace.to_ts-admin-travel-service.duration_mean   0.135750, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object,                         cmdb_id       log_name      p95
0              ts-basic-service  log.row_count  2575.00
1            ts-travel2-service  log.row_count  2401.20
2               ts-seat-service  log.row_count  1909.60
3             ts-travel-service  log.row_count  1890.20
4              ts-route-service  log.row_count  1037.80
5        ts-order-other-service  log.row_count   476.80
6              ts-order-service  log.row_count   421.80
7               ts-auth-service  log.row_count   346.00
8               ts-ui-dashboard  log.row_count   290.40
9              ts-price-service  log.row_count   284.00
10          ts-preserve-service  log.row_count   210.50
11    ts-preserve-other-service  log.row_count   108.75
12              ts-food-service  log.row_count    80.00
13          ts-security-service  log.row_count    72.00
14           ts-consign-service  log.row_count    70.00
15          ts-contacts-service  log.row_count    54.00
16  ts-admin-basic-info-service  log.row_count    30.00
17      ts-admin-travel-service  log.row_count    30.00
18          ts-food-map-service  log.row_count    24.00
19    ts-inside-payment-service  log.row_count    20.00,                   timestamp                      cmdb_id         log_name  value
0 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service  log.error_count      0
1 2024-01-24 10:29:00+00:00  ts-admin-basic-info-service    log.row_count     22
2 2024-01-24 10:29:00+00:00      ts-admin-travel-service  log.error_count      0
3 2024-01-24 10:29:00+00:00      ts-admin-travel-service    log.row_count     21
4 2024-01-24 10:29:00+00:00         ts-assurance-service  log.error_count      0, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Load metric.csv (reuse variable name)
metric_df = pd.read_csv("metric.csv")

# Parse timestamps as UTC (Unix seconds)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric series (before any filtering)
global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric rows to the incident window (inclusive)
start = pd.to_datetime("2024-01-24 10:26:14", utc=True)
end   = pd.to_datetime("2024-01-24 10:56:14", utc=True)
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Merge global P95 into the windowed data so each row knows its global threshold
window_df = window_df.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Aggregate per (cmdb_id, kpi_name) present in the window
def agg_group(g):
    total_count = len(g)
    gp95 = g['global_p95'].iloc[0]
    max_value = g['value'].max(skipna=True)
    # Count anomalies where value >= global_p95 (treat NaN gp95 as no anomaly)
    if pd.isna(gp95):
        anomaly_count = 0
        earliest = pd.NaT
    else:
        anomalies = g[g['value'] >= gp95]
        anomaly_count = int(anomalies.shape[0])
        earliest = anomalies['timestamp'].min() if anomaly_count > 0 else pd.NaT
    # severity
    if pd.isna(gp95) or gp95 == 0:
        severity = None
    else:
        severity = float(max_value) / float(gp95) if not pd.isna(max_value) else None
    return pd.Series({
        'total_count': int(total_count),
        'anomaly_count': int(anomaly_count),
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': max_value,
        'global_p95': gp95,
        'severity': severity
    })

agg_df = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=True)
    .apply(agg_group)
    .reset_index()
)

# 4) Filter only rows where anomaly_count > 0, sort by severity desc, limit top 20
agg_filtered = agg_df[agg_df['anomaly_count'] > 0].copy()
# Convert earliest timestamp to UTC-aware ISO strings for compact display (keep as datetime in var)
agg_filtered['earliest_anomaly_timestamp'] = pd.to_datetime(agg_filtered['earliest_anomaly_timestamp'], utc=True)
agg_filtered_sorted = agg_filtered.sort_values(by=['severity'], ascending=False).head(20).reset_index(drop=True)

# 5) Prepare up to 3 sample metric rows for the top 1-3 component-KPIs to inspect consecutive points
top_n = min(3, agg_filtered_sorted.shape[0])
sample_rows = {}
for i in range(top_n):
    row = agg_filtered_sorted.loc[i]
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    # get up to 5 consecutive points around anomalies: simply take the windowed rows for this group sorted by timestamp and show up to 5
    grp = window_df[(window_df['cmdb_id'] == cmdb) & (window_df['kpi_name'] == kpi)].sort_values('timestamp')
    sample_rows[f'sample_top_{i+1}'] = grp[['timestamp','cmdb_id','kpi_name','value']].head(5).reset_index(drop=True)

# Compact outputs: aggregated table and the sample rows (up to 3)
agg_filtered_sorted[['cmdb_id','kpi_name','total_count','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity']], \
sample_rows.get('sample_top_1', pd.DataFrame()), sample_rows.get('sample_top_2', pd.DataFrame()), sample_rows.get('sample_top_3', pd.DataFrame())

```
Out[2]:
```
summary = (
"Summary of metric anomalies in window 2024-01-24 10:26:14 → 2024-01-24 10:56:14 UTC:\n\n"
"- Method: global P95 was computed using the entire metric series (no time filtering). "
"Metrics were then filtered to the incident window and compared to their global P95.\n\n"
"- Aggregated results (only rows with anomaly_count > 0 were kept; sorted by severity; top 20 shown):\n"
"  Top 3 (by severity):\n"
"  1) ts-route-service | diskio\n"
"     - total_count = 25, anomaly_count = 2\n"
"     - earliest_anomaly_timestamp = 2024-01-24 10:29:00+00:00\n"
"     - max_value_in_window = 214432.784358\n"
"     - global_p95 = 61377.752939\n"
"     - severity = 3.493656\n\n"
"  2) ts-food-map-service | cpu\n"
"     - total_count = 25, anomaly_count = 2\n"
"     - earliest_anomaly_timestamp = 2024-01-24 10:39:00+00:00\n"
"     - max_value_in_window = 2.622558\n"
"     - global_p95 = 0.790120\n"
"     - severity = 3.319190\n\n"
"  3) ts-food-map-service | latency-90\n"
"     - total_count = 25, anomaly_count = 2\n"
"     - earliest_anomaly_timestamp = 2024-01-24 10:39:00+00:00\n"
"     - max_value_in_window = 0.027722\n"
"     - global_p95 = 0.010760\n"
"     - severity = 2.576436\n\n"
"- Notes: the result contained 20 aggregated rows with anomaly_count>0 (only top 20 presented). "
"Most listed groups have total_count = 25 and anomaly_count = 2 in this window.\n\n"
"- Sample metric points (up to 5) returned for the top groups to inspect consecutive points:\n"
"  ts-route-service | diskio (first 5 rows in window):\n"
"    2024-01-24 10:29:00+00:00 — 214432.784358  (the spike)\n"
"    2024-01-24 10:30:00+00:00 — 43517.723004\n"
"    2024-01-24 10:31:00+00:00 — 52285.715351\n"
"    2024-01-24 10:32:00+00:00 — 51432.997989\n"
"    2024-01-24 10:33:00+00:00 — 62099.957344\n\n"
"  ts-food-map-service | cpu (first 5 rows in window):\n"
"    2024-01-24 10:29:00+00:00 — 0.458261\n"
"    2024-01-24 10:30:00+00:00 — 0.439748\n"
"    2024-01-24 10:31:00+00:00 — 0.442064\n"
"    2024-01-24 10:32:00+00:00 — 0.411864\n"
"    2024-01-24 10:33:00+00:00 — 0.699210\n\n"
"  ts-food-map-service | latency-90 (first 5 rows in window):\n"
"    2024-01-24 10:29:00+00:00 — 0.009425\n"
"    2024-01-24 10:30:00+00:00 — 0.009531\n"
"    2024-01-24 10:31:00+00:00 — 0.009330\n"
"    2024-01-24 10:32:00+00:00 — 0.009274\n"
"    2024-01-24 10:33:00+00:00 — 0.009501\n\n"
"- Conclusion / recommended focus:\n"
"  1) Investigate ts-route-service disk I/O around 10:29 UTC — a very large diskio spike (severity ~3.49 vs historical P95).\n"
"  2) Investigate ts-food-map-service around ~10:39 UTC for CPU and latency anomalies (both exceed their global P95 by factors >2.5–3.3).\n\n"
"If you want, I can export the full top-20 aggregated table or show the exact rows around each earliest_anomaly_timestamp for deeper inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  total_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  severity
0              ts-route-service      diskio           25              2  2024-01-24 10:29:00+00:00        214432.784358  61377.752939  3.493656
1           ts-food-map-service         cpu           25              2  2024-01-24 10:39:00+00:00             2.622558      0.790120  3.319190
2           ts-food-map-service  latency-90           25              2  2024-01-24 10:39:00+00:00             0.027722      0.010760  2.576436
3              ts-order-service  latency-90           25              2  2024-01-24 10:33:00+00:00             0.156429      0.062199  2.515001
4         ts-ticketinfo-service         cpu           25              2  2024-01-24 10:29:00+00:00            12.002676      5.973589  2.009291
5              ts-price-service         cpu           25              2  2024-01-24 10:29:00+00:00             1.795773      0.909405  1.974668
6     ts-inside-payment-service         cpu           25              2  2024-01-24 10:39:00+00:00             7.980984      4.330309  1.843052
7            ts-consign-service         cpu           25              2  2024-01-24 10:36:00+00:00             0.882816      0.518145  1.703800
8            ts-payment-service    workload           25              2  2024-01-24 10:29:00+00:00             0.270761      0.166080  1.630304
9           ts-preserve-service         cpu           25              2  2024-01-24 10:31:00+00:00             1.266697      0.829697  1.526699
10         ts-assurance-service         cpu           25              2  2024-01-24 10:39:00+00:00             0.565670      0.379420  1.490881
11             ts-order-service         cpu           25              2  2024-01-24 10:35:00+00:00             3.163770      2.299883  1.375622
12  ts-admin-basic-info-service         cpu           25              2  2024-01-24 10:36:00+00:00             2.713006      1.998653  1.357417
13              ts-auth-service  latency-90           25              2  2024-01-24 10:44:00+00:00             1.034271      0.782580  1.321616
14          ts-contacts-service  latency-90           25              2  2024-01-24 10:36:00+00:00             0.012510      0.009786  1.278358
15             ts-order-service  latency-50           25              2  2024-01-24 10:41:00+00:00             0.013756      0.011050  1.244936
16    ts-inside-payment-service    workload           25              2  2024-01-24 10:29:00+00:00             0.451150      0.365930  1.232886
17           ts-payment-service         cpu           25              2  2024-01-24 10:38:00+00:00             1.225455      0.995578  1.230898
18              ts-user-service         cpu           25              2  2024-01-24 10:32:00+00:00             0.363562      0.297280  1.222963
19      ts-admin-travel-service         cpu           25              2  2024-01-24 10:34:00+00:00             0.534558      0.438527  1.218986,                   timestamp           cmdb_id kpi_name          value
0 2024-01-24 10:29:00+00:00  ts-route-service   diskio  214432.784358
1 2024-01-24 10:30:00+00:00  ts-route-service   diskio   43517.723004
2 2024-01-24 10:31:00+00:00  ts-route-service   diskio   52285.715351
3 2024-01-24 10:32:00+00:00  ts-route-service   diskio   51432.997989
4 2024-01-24 10:33:00+00:00  ts-route-service   diskio   62099.957344,                   timestamp              cmdb_id kpi_name     value
0 2024-01-24 10:29:00+00:00  ts-food-map-service      cpu  0.458261
1 2024-01-24 10:30:00+00:00  ts-food-map-service      cpu  0.439748
2 2024-01-24 10:31:00+00:00  ts-food-map-service      cpu  0.442064
3 2024-01-24 10:32:00+00:00  ts-food-map-service      cpu  0.411864
4 2024-01-24 10:33:00+00:00  ts-food-map-service      cpu  0.699210,                   timestamp              cmdb_id    kpi_name     value
0 2024-01-24 10:29:00+00:00  ts-food-map-service  latency-90  0.009425
1 2024-01-24 10:30:00+00:00  ts-food-map-service  latency-90  0.009531
2 2024-01-24 10:31:00+00:00  ts-food-map-service  latency-90  0.009330
3 2024-01-24 10:32:00+00:00  ts-food-map-service  latency-90  0.009274
4 2024-01-24 10:33:00+00:00  ts-food-map-service  latency-90  0.009501)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Load trace.csv
trace_df = pd.read_csv("trace.csv")

# Parse timestamps as UTC (Unix seconds) and ensure numeric values
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Compute global P95 per (cmdb_id, trace_name) using the full series (no filtering)
trace_global_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Define incident window (inclusive)
start = pd.to_datetime("2024-01-24 10:26:14", utc=True)
end   = pd.to_datetime("2024-01-24 10:56:14", utc=True)

# Filter trace rows to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# Merge global P95 into windowed data
trace_window = trace_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# Aggregate per (cmdb_id, trace_name) present in the window
def agg_trace_group(g):
    total_count = int(len(g))
    gp95 = g['global_p95'].iloc[0]
    max_value = g['value'].max(skipna=True)
    if pd.isna(gp95):
        anomaly_count = 0
        earliest = pd.NaT
    else:
        anomalies = g[g['value'] >= gp95]
        anomaly_count = int(anomalies.shape[0])
        earliest = anomalies['timestamp'].min() if anomaly_count > 0 else pd.NaT
    if pd.isna(gp95) or gp95 == 0:
        severity = None
    else:
        severity = float(max_value) / float(gp95) if not pd.isna(max_value) else None
    return pd.Series({
        'total_count': total_count,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': max_value,
        'global_p95': gp95,
        'severity': severity
    })

trace_agg = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], as_index=True)
    .apply(agg_trace_group)
    .reset_index()
)

# Filter only rows where anomaly_count > 0, sort by severity desc, limit to top 20
trace_agg_filtered = trace_agg[trace_agg['anomaly_count'] > 0].copy()
trace_agg_filtered['earliest_anomaly_timestamp'] = pd.to_datetime(trace_agg_filtered['earliest_anomaly_timestamp'], utc=True)
trace_agg_top20 = trace_agg_filtered.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# Additionally, return up to 20 trace rows from the incident window whose trace_name contains any of the service names:
services = ['ts-route-service', 'ts-food-map-service', 'ts-order-service']
pattern = '|'.join([s.replace('-', r'\-') for s in services])  # escape hyphens for regex
sample_trace_rows = (
    trace_window[trace_window['trace_name'].str.contains(pattern, regex=True, na=False)]
    .sort_values('timestamp')
    [['timestamp','cmdb_id','trace_name','value']]
    .head(20)
    .reset_index(drop=True)
)

# Compact outputs
trace_agg_top20[['cmdb_id','trace_name','total_count','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity']], sample_trace_rows

```
Out[3]:
```
summary = (
"Trace anomaly summary for window 2024-01-24 10:26:14 → 2024-01-24 10:56:14 UTC:\n\n"
"- Method: global P95 per (cmdb_id,trace_name) was computed using the full trace series, then windowed traces were compared to those global P95 thresholds.\n\n"
"- Top anomalies (only groups with anomaly_count>0 shown; sorted by severity):\n"
"  1) ts-food-service → trace.to_ts-food-map-service.duration_p95\n"
"     - total_count = 25, anomaly_count = 2\n"
"     - earliest anomaly = 2024-01-24 10:35:00+00:00\n"
"     - max value = 0.048742, global_p95 = 0.005962, severity ≈ 8.18\n"
"  2) ts-food-map-service → trace.from_ts-food-service.duration_p95\n"
"     - same counts/timestamps/values as (1); severity ≈ 8.18 (reciprocal interaction spike between food-service and food-map-service at ~10:35 UTC)\n"
"  3) ts-preserve-service → trace.to_ts-order-service.duration_p95\n"
"     - total_count = 23, anomaly_count = 2, earliest = 2024-01-24 10:40:00+00:00\n"
"     - max = 0.064854, global_p95 = 0.024653, severity ≈ 2.63\n"
"  4) ts-order-service → trace.from_ts-preserve-service.duration_p95\n"
"     - mirrored pair to (3), severity ≈ 2.63\n"
"  5) ts-order-service ↔ ts-seat-service duration_p95 pairs (severity ≈ 2.21)\n\n"
"- Other notable findings: multiple services show paired trace.duration_p95 spikes (from/to pairs), indicating latency increases on service-to-service calls rather than isolated single-side spikes.\n\n"
"- Sample trace rows (earliest 20 in window that reference ts-route-service, ts-food-map-service, ts-order-service):\n"
"  - include many entries at 2024-01-24 10:29:00+00:00 such as route-service row_count spikes (e.g., 1521), various duration_p95 and duration_mean entries, and several order-related traces. These samples show both traffic volume (row_count) and duration spikes at different timestamps in the window.\n\n"
"- Recommended next steps:\n"
"  1) Immediately investigate ts-food-service ↔ ts-food-map-service around 10:35 UTC — very large duration_p95 spike (~8x historical P95).\n"
"  2) Investigate ts-preserve ↔ ts-order and ts-order ↔ ts-seat interactions around 10:35–10:40 UTC for elevated call durations (severity ~2–2.6x).\n"
"  3) Correlate these trace spikes with metric (CPU/diskIO) and log anomalies for the same services/timestamps to determine if latency is caused by resource pressure or increased request volume.\n\n"
"If you want, I can (a) show the full top-20 aggregated trace anomalies table, or (b) extract the exact trace rows around each earliest_anomaly_timestamp for deeper inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id                                         trace_name  total_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity
0               ts-food-service          trace.to_ts-food-map-service.duration_p95           25              2  2024-01-24 10:35:00+00:00             0.048742    0.005962  8.175871
1           ts-food-map-service            trace.from_ts-food-service.duration_p95           25              2  2024-01-24 10:35:00+00:00             0.048742    0.005962  8.175871
2           ts-preserve-service             trace.to_ts-order-service.duration_p95           23              2  2024-01-24 10:40:00+00:00             0.064854    0.024653  2.630656
3              ts-order-service        trace.from_ts-preserve-service.duration_p95           23              2  2024-01-24 10:40:00+00:00             0.064854    0.024653  2.630656
4              ts-order-service            trace.from_ts-seat-service.duration_p95           25              2  2024-01-24 10:40:00+00:00             0.040427    0.018281  2.211440
5               ts-seat-service             trace.to_ts-order-service.duration_p95           25              2  2024-01-24 10:40:00+00:00             0.040427    0.018281  2.211440
6           ts-contacts-service  trace.from_ts-preserve-other-service.duration_p95           22              2  2024-01-24 10:35:00+00:00             0.012263    0.005811  2.110396
7     ts-preserve-other-service          trace.to_ts-contacts-service.duration_p95           22              2  2024-01-24 10:35:00+00:00             0.012263    0.005811  2.110396
8           ts-food-map-service          trace.to_ts-food-map-service.duration_p95           25              2  2024-01-24 10:39:00+00:00             0.005083    0.002547  1.995579
9           ts-food-map-service        trace.from_ts-food-map-service.duration_p95           25              2  2024-01-24 10:39:00+00:00             0.005083    0.002547  1.995579
10                         root         trace.to_ts-assurance-service.duration_p95           24              2  2024-01-24 10:35:00+00:00             0.009036    0.004613  1.958605
11         ts-assurance-service                       trace.from_root.duration_p95           24              2  2024-01-24 10:35:00+00:00             0.009036    0.004613  1.958605
12              ts-food-service         trace.to_ts-food-map-service.duration_mean           25              2  2024-01-24 10:38:00+00:00             0.010310    0.005568  1.851729
13          ts-food-map-service           trace.from_ts-food-service.duration_mean           25              2  2024-01-24 10:38:00+00:00             0.010310    0.005568  1.851729
14          ts-preserve-service                 trace.to_ts-food-service.row_count           21              3  2024-01-24 10:35:00+00:00             7.000000    4.000000  1.750000
15              ts-food-service           trace.from_ts-preserve-service.row_count           21              3  2024-01-24 10:35:00+00:00             7.000000    4.000000  1.750000
16           ts-consign-service                       trace.from_root.duration_p95           25              2  2024-01-24 10:31:00+00:00             0.053952    0.032502  1.659956
17                         root           trace.to_ts-consign-service.duration_p95           25              2  2024-01-24 10:31:00+00:00             0.053952    0.032502  1.659956
18  ts-admin-basic-info-service                       trace.from_root.duration_p95           25              2  2024-01-24 10:43:00+00:00             0.025555    0.015433  1.655912
19                         root  trace.to_ts-admin-basic-info-service.duration_p95           25              2  2024-01-24 10:43:00+00:00             0.025555    0.015433  1.655912,                    timestamp              cmdb_id                                 trace_name        value
0  2024-01-24 10:29:00+00:00                 root    trace.to_ts-order-service.duration_mean     0.546492
1  2024-01-24 10:29:00+00:00      ts-seat-service        trace.to_ts-order-service.row_count    84.000000
2  2024-01-24 10:29:00+00:00      ts-seat-service       trace.to_ts-order-service.error_rate     0.000000
3  2024-01-24 10:29:00+00:00      ts-seat-service     trace.to_ts-order-service.duration_p95     0.015606
4  2024-01-24 10:29:00+00:00      ts-seat-service    trace.to_ts-order-service.duration_mean     0.006155
5  2024-01-24 10:29:00+00:00     ts-route-service        trace.to_ts-route-service.row_count  1521.000000
6  2024-01-24 10:29:00+00:00     ts-route-service       trace.to_ts-route-service.error_rate     0.000000
7  2024-01-24 10:29:00+00:00  ts-security-service     trace.to_ts-order-service.duration_p95     0.011437
8  2024-01-24 10:29:00+00:00     ts-route-service     trace.to_ts-route-service.duration_p95     0.001560
9  2024-01-24 10:29:00+00:00     ts-route-service      trace.from_ts-route-service.row_count  1521.000000
10 2024-01-24 10:29:00+00:00     ts-route-service     trace.from_ts-route-service.error_rate     0.000000
11 2024-01-24 10:29:00+00:00     ts-route-service   trace.from_ts-route-service.duration_p95     0.001560
12 2024-01-24 10:29:00+00:00     ts-route-service  trace.from_ts-route-service.duration_mean     0.001087
13 2024-01-24 10:29:00+00:00  ts-preserve-service        trace.to_ts-order-service.row_count     2.000000
14 2024-01-24 10:29:00+00:00  ts-preserve-service       trace.to_ts-order-service.error_rate     0.000000
15 2024-01-24 10:29:00+00:00     ts-route-service    trace.to_ts-route-service.duration_mean     0.001087
16 2024-01-24 10:29:00+00:00  ts-security-service       trace.to_ts-order-service.error_rate     0.000000
17 2024-01-24 10:29:00+00:00  ts-security-service        trace.to_ts-order-service.row_count     2.000000
18 2024-01-24 10:29:00+00:00   ts-station-service  trace.from_ts-order-service.duration_mean     0.529996
19 2024-01-24 10:29:00+00:00   ts-travel2-service        trace.to_ts-route-service.row_count   427.000000)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse variable names where sensible; load log and error logs fresh
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC and ensure numeric 'value' for logs
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Compute global P95 per (cmdb_id, log_name) using the full series (before filtering)
log_global_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Define incident window (inclusive)
start = pd.to_datetime("2024-01-24 10:26:14", utc=True)
end   = pd.to_datetime("2024-01-24 10:56:14", utc=True)

# Filter log rows to incident window
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# Merge global p95 into windowed logs
log_window = log_window.merge(log_global_p95, on=['cmdb_id', 'log_name'], how='left')

# Aggregate per (cmdb_id, log_name) present in the window
def agg_log_group(g):
    total_count = int(len(g))
    gp95 = g['global_p95'].iloc[0]
    max_value = g['value'].max(skipna=True)
    if pd.isna(gp95):
        anomaly_count = 0
        earliest = pd.NaT
    else:
        anomalies = g[g['value'] >= gp95]
        anomaly_count = int(anomalies.shape[0])
        earliest = anomalies['timestamp'].min() if anomaly_count > 0 else pd.NaT
    if pd.isna(gp95) or gp95 == 0:
        severity = None
    else:
        severity = float(max_value) / float(gp95) if not pd.isna(max_value) else None
    return pd.Series({
        'total_count': total_count,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': max_value,
        'global_p95': gp95,
        'severity': severity
    })

log_agg = (
    log_window
    .groupby(['cmdb_id', 'log_name'], as_index=True)
    .apply(agg_log_group)
    .reset_index()
)

# Filter only rows where anomaly_count > 0, sort by severity desc, limit top 20
log_agg_filtered = log_agg[log_agg['anomaly_count'] > 0].copy()
log_agg_filtered['earliest_anomaly_timestamp'] = pd.to_datetime(log_agg_filtered['earliest_anomaly_timestamp'], utc=True)
log_agg_top20 = log_agg_filtered.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# Sample up to 10 log rows from the incident window for selected services (earliest rows)
services = ['ts-food-map-service', 'ts-food-service', 'ts-route-service', 'ts-order-service', 'ts-preserve-service']
sample_log_rows = (
    log_window[log_window['cmdb_id'].isin(services)]
    .sort_values('timestamp')
    [['timestamp','cmdb_id','log_name','value']]
    .head(10)
    .reset_index(drop=True)
)

# Report presence of rows in error_logs.csv overall and within the window
error_overall_count = int(len(error_df))
error_window_count = int(error_df[(error_df['timestamp'] >= start) & (error_df['timestamp'] <= end)].shape[0])
# up to 5 sample error rows (earliest) for context
error_sample_head = error_df.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(5).reset_index(drop=True)

# Compact outputs
log_agg_top20[['cmdb_id','log_name','total_count','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity']], sample_log_rows, error_overall_count, error_window_count, error_sample_head

```
Out[4]:
```
Summary (plain English):

- What I did: computed global P95 per (cmdb_id,log_name) using the entire log series (no time filtering), then examined log rows in the incident window 2024-01-24 10:26:14 → 2024-01-24 10:56:14 UTC and flagged points >= that global P95.

- Top log anomalies (only groups with anomaly_count > 0; sorted by severity):
  1) ts-consign-mongo | log.row_count
     - total_count = 24, anomaly_count = 24
     - earliest anomaly = 2024-01-24 10:30:00+00:00
     - max = 2, global_p95 = 1.00, severity = 2.00
  2) ts-consign-service | log.row_count
     - total_count = 25, anomaly_count = 3, earliest = 2024-01-24 10:33:00+00:00
     - max = 84, global_p95 = 70.00, severity = 1.20
  3) ts-assurance-service | log.row_count
     - total_count = 24, anomaly_count = 2, earliest = 2024-01-24 10:32:00+00:00
     - max = 15, global_p95 ≈ 12.85, severity ≈ 1.17
  4) ts-food-service | log.row_count
     - total_count = 25, anomaly_count = 3, earliest = 2024-01-24 10:35:00+00:00
     - max = 91, global_p95 = 80.00, severity ≈ 1.14
  5) ts-preserve-service | log.row_count
     - total_count = 23, anomaly_count = 2, earliest = 2024-01-24 10:32:00+00:00
     - max = 239, global_p95 = 210.50, severity ≈ 1.14
  (other services appear with milder severity ~1.01–1.04; full top-20 was computed.)

- Requested services (compact per-service highlights from the aggregated table):
  - ts-food-map-service | log.row_count: total_count = 25, anomaly_count = 3, earliest = 2024-01-24 10:35:00+00:00, severity ≈ 1.04
  - ts-food-service     | log.row_count: total_count = 25, anomaly_count = 3, earliest = 2024-01-24 10:35:00+00:00, severity ≈ 1.14
  - ts-route-service    | log.row_count: total_count = 25, anomaly_count = 2, earliest = 2024-01-24 10:31:00+00:00, severity ≈ 1.02
  - ts-order-service    | log.row_count: total_count = 25, anomaly_count = 2, earliest = 2024-01-24 10:31:00+00:00, severity ≈ 1.05
  - ts-preserve-service | log.row_count: total_count = 23, anomaly_count = 2, earliest = 2024-01-24 10:32:00+00:00, severity ≈ 1.14

- Sample log rows returned (earliest rows in window for the requested services):
  - 2024-01-24 10:29:00+00:00 — ts-food-map-service — log.row_count = 18 (log.error_count = 0)
  - 2024-01-24 10:29:00+00:00 — ts-food-service     — log.row_count = 58 (log.error_count = 0)
  - 2024-01-24 10:29:00+00:00 — ts-order-service    — log.row_count = 308 (log.error_count = 0)
  - 2024-01-24 10:29:00+00:00 — ts-preserve-service — log.row_count = 57 (log.error_count = 0)
  - 2024-01-24 10:29:00+00:00 — ts-route-service    — log.row_count = 760 (log.error_count = 0)

- error_logs.csv: empty overall and in the incident window (no error log rows available).

- Interpretation / suggested next steps:
  - Many services show modest increases in log.row_count above their historical P95 (severity mostly in the 1.01–1.20 range). This suggests traffic/log volume spikes rather than catastrophic failures for most services.
  - Highest log severity is on ts-consign-mongo and ts-consign-service (severity up to 2.0 and ~1.2); ts-food-service and ts-preserve-service show notable increases (~1.14).
  - Correlate these log count spikes with the metric and trace anomalies previously found (e.g., diskio/cpu/latency spikes) to identify whether increased traffic, resource saturation, or cross-service latency is the root cause.
  - No error log entries are available to indicate explicit application errors in this dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id       log_name  total_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity
0            ts-consign-mongo  log.row_count           24             24  2024-01-24 10:30:00+00:00                    2        1.00  2.000000
1          ts-consign-service  log.row_count           25              3  2024-01-24 10:33:00+00:00                   84       70.00  1.200000
2        ts-assurance-service  log.row_count           24              2  2024-01-24 10:32:00+00:00                   15       12.85  1.167315
3             ts-food-service  log.row_count           25              3  2024-01-24 10:35:00+00:00                   91       80.00  1.137500
4         ts-preserve-service  log.row_count           23              2  2024-01-24 10:32:00+00:00                  239      210.50  1.135392
5         ts-contacts-service  log.row_count           24              3  2024-01-24 10:32:00+00:00                   60       54.00  1.111111
6         ts-security-service  log.row_count           24              3  2024-01-24 10:32:00+00:00                   80       72.00  1.111111
7            ts-order-service  log.row_count           25              2  2024-01-24 10:31:00+00:00                  442      421.80  1.047890
8         ts-food-map-service  log.row_count           25              3  2024-01-24 10:35:00+00:00                   25       24.00  1.041667
9             ts-auth-service  log.row_count           25              3  2024-01-24 10:30:00+00:00                  356      346.00  1.028902
10  ts-preserve-other-service  log.row_count           22              2  2024-01-24 10:37:00+00:00                  111      108.75  1.020690
11           ts-route-service  log.row_count           25              2  2024-01-24 10:31:00+00:00                 1058     1037.80  1.019464
12            ts-ui-dashboard  log.row_count           25              2  2024-01-24 10:31:00+00:00                  295      290.40  1.015840
13            ts-seat-service  log.row_count           25              2  2024-01-24 10:31:00+00:00                 1938     1909.60  1.014872
14          ts-travel-service  log.row_count           25              2  2024-01-24 10:31:00+00:00                 1913     1890.20  1.012062
15           ts-basic-service  log.row_count           25              2  2024-01-24 10:31:00+00:00                 2597     2575.00  1.008544
16         ts-travel2-service  log.row_count           25              2  2024-01-24 10:30:00+00:00                 2421     2401.20  1.008246
17           ts-price-service  log.row_count           25              3  2024-01-24 10:31:00+00:00                  286      284.00  1.007042
18     ts-order-other-service  log.row_count           25              2  2024-01-24 10:30:00+00:00                  480      476.80  1.006711
19         ts-assurance-mongo  log.row_count           24             24  2024-01-24 10:30:00+00:00                    1        1.00  1.000000,                   timestamp              cmdb_id         log_name  value
0 2024-01-24 10:29:00+00:00  ts-food-map-service  log.error_count      0
1 2024-01-24 10:29:00+00:00  ts-food-map-service    log.row_count     18
2 2024-01-24 10:29:00+00:00      ts-food-service  log.error_count      0
3 2024-01-24 10:29:00+00:00      ts-food-service    log.row_count     58
4 2024-01-24 10:29:00+00:00     ts-order-service  log.error_count      0
5 2024-01-24 10:29:00+00:00     ts-order-service    log.row_count    308
6 2024-01-24 10:29:00+00:00  ts-preserve-service  log.error_count      0
7 2024-01-24 10:29:00+00:00  ts-preserve-service    log.row_count     57
8 2024-01-24 10:29:00+00:00     ts-route-service  log.error_count      0
9 2024-01-24 10:29:00+00:00     ts-route-service    log.row_count    760, 0, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd

# Load files (re-loading to ensure fresh state)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Parse timestamps as UTC and ensure numeric values
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Compute global P95 per (cmdb_id, kpi_name) using the full metric series (before filtering)
metric_global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Incident window (inclusive)
start = pd.to_datetime("2024-01-24 10:26:14", utc=True)
end   = pd.to_datetime("2024-01-24 10:56:14", utc=True)

# Filter metric and trace data to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
trace_window  = trace_df[(trace_df['timestamp']  >= start) & (trace_df['timestamp']  <= end)].copy()

# Focus on cmdb_id == 'ts-food-map-service'
cmdb = 'ts-food-map-service'

# 1) Global P95 for each kpi_name for this cmdb_id (from the full-series global_p95)
p95s_food_map = metric_global_p95[metric_global_p95['cmdb_id'] == cmdb].sort_values('kpi_name').reset_index(drop=True)

# 2) Return metric rows in window for specified kpis (if present)
kpis_of_interest = ['cpu', 'latency-90', 'latency-50', 'socket', 'mem', 'diskio']
metric_food_map_window = (
    metric_window[
        (metric_window['cmdb_id'] == cmdb) &
        (metric_window['kpi_name'].isin(kpis_of_interest))
    ]
    .loc[:, ['timestamp','cmdb_id','kpi_name','value']]
    .sort_values('timestamp')
    .head(50)
    .reset_index(drop=True)
)

# Trace rows in window where trace_name contains any of the specified substrings
trace_pattern = r'food-map-service|to_ts-food-map-service|from_ts-food-service'
trace_food_map_window = (
    trace_window[trace_window['trace_name'].str.contains(trace_pattern, regex=True, na=False)]
    .loc[:, ['timestamp','cmdb_id','trace_name','value']]
    .sort_values('timestamp')
    .head(50)
    .reset_index(drop=True)
)

# Compact outputs
p95s_food_map, metric_food_map_window, trace_food_map_window

```
Out[5]:
```
summary = (
"ts-food-map-service snapshot for window 2024-01-24 10:26:14 → 2024-01-24 10:56:14 UTC:\n\n"
"1) Global P95s (computed on the full metric series):\n"
"   - cpu: 0.79012\n"
"   - diskio: 59964.84\n"
"   - latency-50: 0.0069528\n"
"   - latency-90: 0.01076\n"
"   - mem: 256,788,800.0\n"
"   - socket: 9.3\n"
"   - workload: 1.537216\n\n"
"2) Metric rows in the incident window (sample / up to 50 rows):\n"
"   - CPU values in the window sample are around 0.44–0.70 (several points shown near 0.45–0.70).\n"
"   - diskio values shown around ~54,396 (one row ~54k) in the sample.\n"
"   - latency-50/90 values in the sample are ~0.0069 and ~0.0094 respectively.\n"
"   - mem values are ~2.5616e+08 (consistent high memory values in the sample).\n"
"   - socket values in the sample are 8.8 and 9.0.\n"
"   (These sample rows are sorted by timestamp; full returned table is limited to 50 rows.)\n\n"
"3) Trace rows in the window matching food-map / food-service interactions (sample / up to 50 rows):\n"
"   - Contains calls both from/to food-map-service and from_ts-food-service entries.\n"
"   - Examples: duration_mean entries (~0.0015–0.01), duration_p95 entries (~0.0053–0.0149), and row_count spikes (e.g., 72 at 10:30, 18 at 10:29).\n"
"   - These traces show both low mean latencies and occasional elevated duration_p95 values and row_count activity.\n\n"
"Overall takeaway: ts-food-map-service has clear historical P95 thresholds for CPU (~0.79), latency-90 (~0.0108), and large memory (~2.57e8). The sampled metric rows in the incident window mostly show CPU and latency values below those P95s in the earliest returned rows, but traces show occasional elevated duration_p95 and row_count activity. (Previous anomaly analysis also indicated spikes around ~10:35–10:39 UTC for food-service/food-map-service interactions.)"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id    kpi_name    global_p95
0  ts-food-map-service         cpu  7.901199e-01
1  ts-food-map-service      diskio  5.996484e+04
2  ts-food-map-service  latency-50  6.952784e-03
3  ts-food-map-service  latency-90  1.075999e-02
4  ts-food-map-service         mem  2.567888e+08
5  ts-food-map-service      socket  9.300000e+00
6  ts-food-map-service    workload  1.537216e+00,                    timestamp              cmdb_id    kpi_name         value
0  2024-01-24 10:29:00+00:00  ts-food-map-service         cpu  4.582612e-01
1  2024-01-24 10:29:00+00:00  ts-food-map-service      diskio  5.439630e+04
2  2024-01-24 10:29:00+00:00  ts-food-map-service  latency-50  6.895016e-03
3  2024-01-24 10:29:00+00:00  ts-food-map-service  latency-90  9.424673e-03
4  2024-01-24 10:29:00+00:00  ts-food-map-service         mem  2.561613e+08
..                       ...                  ...         ...           ...
45 2024-01-24 10:36:00+00:00  ts-food-map-service  latency-90  9.463906e-03
46 2024-01-24 10:36:00+00:00  ts-food-map-service         mem  2.561937e+08
47 2024-01-24 10:36:00+00:00  ts-food-map-service      socket  8.800000e+00
48 2024-01-24 10:37:00+00:00  ts-food-map-service      socket  9.000000e+00
49 2024-01-24 10:37:00+00:00  ts-food-map-service         mem  2.562175e+08

[50 rows x 4 columns],                    timestamp              cmdb_id                                    trace_name      value
0  2024-01-24 10:29:00+00:00  ts-food-map-service  trace.from_ts-food-map-service.duration_mean   0.001730
1  2024-01-24 10:29:00+00:00    ts-travel-service          trace.from_ts-food-service.row_count  18.000000
2  2024-01-24 10:29:00+00:00    ts-travel-service         trace.from_ts-food-service.error_rate   0.000000
3  2024-01-24 10:29:00+00:00    ts-travel-service       trace.from_ts-food-service.duration_p95   0.014892
4  2024-01-24 10:29:00+00:00    ts-travel-service      trace.from_ts-food-service.duration_mean   0.009967
..                       ...                  ...                                           ...        ...
45 2024-01-24 10:30:00+00:00  ts-food-map-service      trace.from_ts-food-map-service.row_count  72.000000
46 2024-01-24 10:30:00+00:00  ts-food-map-service      trace.from_ts-food-service.duration_mean   0.004637
47 2024-01-24 10:30:00+00:00  ts-food-map-service       trace.from_ts-food-service.duration_p95   0.005318
48 2024-01-24 10:30:00+00:00  ts-food-map-service         trace.from_ts-food-service.error_rate   0.000000
49 2024-01-24 10:30:00+00:00  ts-food-map-service  trace.from_ts-food-map-service.duration_mean   0.001540

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```